In [34]:
# Initial clustering experiment
# Step 1: Import required libraries

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pytsg import parse_tsg

print("Libraries loaded successfully")

Libraries loaded successfully


In [ ]:
# Step 2: Load the HyLogger package for 05GJD001

HOLE_ID = "MPWD10"

# Local folder used by the new downloader. This does not depend on Path.cwd().
RAW_DATA_ROOT = Path(
    "C:/Users/DCCN9/OneDrive - UWA/Data Science/"
    "4-Semester July-November 2026/Capstone/Data Crunch/data/downloads/raw"
)

HOLE_DOWNLOAD_DIR = RAW_DATA_ROOT / HOLE_ID
ZIP_FILE = HOLE_DOWNLOAD_DIR / "archive" / f"{HOLE_ID}.zip"
EXTRACTED_DIR = HOLE_DOWNLOAD_DIR / "extracted"
header_file = next(EXTRACTED_DIR.rglob("*_tsg.tsg"), None)
PACKAGE = header_file.parent if header_file else None

if PACKAGE is None:
    raise FileNotFoundError(
        f"The ZIP is at {ZIP_FILE}, but pytsg needs the extracted spectral files.\n"
        f"Expected them under: {EXTRACTED_DIR}"
    )

print("Package found:")
print(PACKAGE)

# Load the HyLogger package
package = parse_tsg.read_package(PACKAGE, read_cras_file=False)

nir = package.nir
tir = package.tir

hole_id = str(nir.sampleheaders["H"].iloc[0])

print("\nHole:", hole_id)
print("Number of readings:", f"{len(nir.spectra):,}")
print("NIR/SWIR wavelengths:", nir.spectra.shape[1])
print("TIR wavelengths:", tir.spectra.shape[1])

FileNotFoundError: The ZIP is at C:\Users\DCCN9\OneDrive - UWA\Data Science\4-Semester July-November 2026\Capstone\Data Crunch\data\downloads\raw\05KCD001\archive\05KCD001.zip, but pytsg needs the extracted spectral files.
Expected them under: C:\Users\DCCN9\OneDrive - UWA\Data Science\4-Semester July-November 2026\Capstone\Data Crunch\data\downloads\raw\05KCD001\extracted

In [ ]:
# Step 3: Get depth and Final Mask for each reading

# Location information
where = nir.sampleheaders.copy()

where = where.rename(columns={
    "D": "depth_m",
    "T": "tray",
    "H": "hole_id"
})

# Scalar information calculated by TSG
scalars = nir.scalars.copy()

# Create a small table to inspect the mask
mask_info = pd.DataFrame({
    "depth_m": where["depth_m"].values,
    "tray": where["tray"].values,
    "Final_Mask": scalars["Final Mask"].values
})

print("Total readings:", len(mask_info))

print("\nFinal Mask counts:")
print(mask_info["Final_Mask"].value_counts(dropna=False))

display(mask_info.head(20))

In [ ]:
# Step 4: Remove only rows where Final Mask = 0

# True for rows we want to keep
keep_mask = mask_info["Final_Mask"].values == 1

# Keep only the corresponding NIR/SWIR spectra
spectra_kept = nir.spectra[keep_mask]

# Keep matching depth/tray information
metadata_kept = mask_info.loc[keep_mask].reset_index(drop=True)

print("Original readings:", f"{len(mask_info):,}")
print("Removed (Final Mask = 0):", f"{(~keep_mask).sum():,}")
print("Kept (Final Mask = 1):", f"{keep_mask.sum():,}")

print("\nSpectra ready after mask filtering:")
print("Rows:", f"{spectra_kept.shape[0]:,}")
print("Wavelengths:", spectra_kept.shape[1])

display(metadata_kept.head(10))

## Check the filtered spectra

After removing the readings where `Final Mask = 0`, we are left with 18,508 spectra.

Before standardising the data or running PCA, we first check whether the remaining spectra contain any missing (`NaN`) or infinite values.

This is important because PCA and K-Means require valid numeric values. This step only checks the data — it does not remove any additional rows.

In [ ]:
# Step 5: Check filtered spectra for missing or invalid numeric values

spectra_array = np.asarray(spectra_kept, dtype=float)

# Check each spectrum (row)
has_nan = np.isnan(spectra_array).any(axis=1)
has_inf = np.isinf(spectra_array).any(axis=1)
has_nonfinite = ~np.isfinite(spectra_array).all(axis=1)

print("Spectra checked:", f"{len(spectra_array):,}")
print("Rows containing NaN:", f"{has_nan.sum():,}")
print("Rows containing Inf:", f"{has_inf.sum():,}")
print("Rows with any non-finite value:", f"{has_nonfinite.sum():,}")

print("\nOverall spectral value range:")
print("Minimum:", np.nanmin(spectra_array))
print("Maximum:", np.nanmax(spectra_array))

## Check reflectance levels

The remaining 18,508 spectra contain no missing or infinite values.

Before PCA and clustering, we inspect the overall reflectance levels of each spectrum. This helps identify unusually bright or extreme spectra that could strongly influence the clustering.

For each reading, we calculate its mean and maximum reflectance. At this stage, no additional rows are removed.

In [ ]:
# Calculate reflectance summaries for each retained spectrum

mean_reflectance = spectra_array.mean(axis=1)
max_reflectance = spectra_array.max(axis=1)

reflectance_summary = pd.DataFrame({
    "depth_m": metadata_kept["depth_m"].values,
    "tray": metadata_kept["tray"].values,
    "mean_reflectance": mean_reflectance,
    "max_reflectance": max_reflectance
})

print("Mean reflectance percentiles:")
print(
    np.percentile(
        mean_reflectance,
        [0, 25, 50, 75, 90, 95, 99, 100]
    )
)

print("\nMaximum reflectance percentiles:")
print(
    np.percentile(
        max_reflectance,
        [0, 25, 50, 75, 90, 95, 99, 100]
    )
)

print("\nHighest-reflectance readings:")
display(
    reflectance_summary
    .sort_values("max_reflectance", ascending=False)
    .head(20)
)

In [ ]:
# Plot maximum reflectance against numeric depth

plot_data = reflectance_summary.copy()

# Make sure depth is numeric
plot_data["depth_m"] = pd.to_numeric(
    plot_data["depth_m"],
    errors="coerce"
)

# Remove any rows where depth could not be converted
plot_data = plot_data.dropna(subset=["depth_m"])

# Sort by depth
plot_data = plot_data.sort_values("depth_m")

print("Depth range:")
print(
    f"{plot_data['depth_m'].min():.2f} m to "
    f"{plot_data['depth_m'].max():.2f} m"
)

plt.figure(figsize=(12, 5))

plt.scatter(
    plot_data["depth_m"],
    plot_data["max_reflectance"],
    s=3,
    alpha=0.25
)

plt.xlabel("Depth (m)")
plt.ylabel("Maximum reflectance")
plt.title(f"Maximum Reflectance by Depth — {hole_id}")

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Standardise the spectral wavelengths

The retained dataset contains 18,508 spectra, with 531 wavelength measurements for each reading.

Before PCA and K-Means, the wavelength features are standardised so that wavelengths with larger variation do not dominate the analysis.

Standardisation is performed separately for each wavelength across all retained spectra. No rows or wavelength columns are removed in this step.

In [ ]:
# Standardise the 531 wavelength features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

spectra_scaled = scaler.fit_transform(spectra_array)

print("Before standardisation:", spectra_array.shape)
print("After standardisation:", spectra_scaled.shape)

print("\nOverall mean after standardisation:",
      round(spectra_scaled.mean(), 6))

print("Overall standard deviation after standardisation:",
      round(spectra_scaled.std(), 6))

## Reduce the spectral dimensions with PCA

Each retained spectrum currently contains 531 wavelength features. Many neighbouring wavelengths are strongly related, so using all 531 directly for clustering would include a lot of redundant information.

Principal Component Analysis (PCA) compresses the spectral data into a smaller number of components while preserving most of the variation in the spectra.

For the initial analysis, PCA is set to retain 95% of the total variance. The number of components required to reach this level is determined automatically from the data.

In [ ]:
# Reduce the 531 wavelength features using PCA

from sklearn.decomposition import PCA

# Keep enough principal components to explain 95% of the variance
pca = PCA(n_components=0.95)

spectra_pca = pca.fit_transform(spectra_scaled)

print("Original spectral shape:", spectra_scaled.shape)
print("PCA shape:", spectra_pca.shape)

print("\nNumber of PCA components retained:",
      pca.n_components_)

print("Total explained variance:",
      round(pca.explained_variance_ratio_.sum() * 100, 2),
      "%")

## Inspect the PCA components

PCA reduced the 531 wavelength features to only 3 principal components while retaining 96.86% of the total variance.

Before using these components for clustering, we inspect how much variance is explained by each component individually.

We also compare the principal components with the mean reflectance of each spectrum. This helps determine whether a component is mainly representing overall brightness or other differences in spectral shape.

In [ ]:
# Inspect the three PCA components

for i, variance in enumerate(pca.explained_variance_ratio_, start=1):
    print(
        f"PC{i}: {variance * 100:.2f}% "
        f"of variance explained"
    )

# Correlation between each PCA component and mean reflectance
print("\nCorrelation with mean reflectance:")

for i in range(spectra_pca.shape[1]):
    correlation = np.corrcoef(
        spectra_pca[:, i],
        mean_reflectance
    )[0, 1]

    print(
        f"PC{i + 1}: {correlation:.3f}"
    )

## Normalise individual spectra with SNV

The initial PCA showed that overall reflectance intensity dominates the first principal component.

To reduce this brightness effect, Standard Normal Variate (SNV) normalisation is applied separately to each spectrum. For every reading, its mean reflectance is subtracted and the result is divided by its standard deviation.

This preserves the relative shape of the spectrum while reducing differences caused mainly by overall brightness.

The original filtered spectra are kept unchanged so that the two approaches can be compared.

In [ ]:
# Apply Standard Normal Variate (SNV) to each spectrum

row_mean = spectra_array.mean(axis=1, keepdims=True)
row_std = spectra_array.std(axis=1, keepdims=True)

# Check that no spectrum has zero variation
zero_std = (row_std[:, 0] == 0)

print("Spectra with zero standard deviation:", zero_std.sum())

# SNV normalisation
spectra_snv = (spectra_array - row_mean) / row_std

print("Original shape:", spectra_array.shape)
print("SNV shape:", spectra_snv.shape)

print("\nMean of first SNV spectrum:",
      round(spectra_snv[0].mean(), 6))

print("Standard deviation of first SNV spectrum:",
      round(spectra_snv[0].std(), 6))

## PCA after SNV normalisation

SNV normalisation was applied successfully to all 18,508 retained spectra.

The next step is to run PCA again on the SNV-normalised spectra. This allows us to check whether the strong brightness effect observed in the first PCA has been reduced.

As before, PCA retains enough components to explain at least 95% of the total variance.

In [ ]:
# Run PCA on the SNV-normalised spectra

pca_snv = PCA(n_components=0.95)

spectra_pca_snv = pca_snv.fit_transform(spectra_snv)

print("SNV spectral shape:", spectra_snv.shape)
print("PCA after SNV shape:", spectra_pca_snv.shape)

print("\nNumber of PCA components retained:",
      pca_snv.n_components_)

print("Total explained variance:",
      round(pca_snv.explained_variance_ratio_.sum() * 100, 2),
      "%")

print("\nVariance explained by each component:")

for i, variance in enumerate(
    pca_snv.explained_variance_ratio_,
    start=1
):
    print(f"PC{i}: {variance * 100:.2f}%")

### Check the brightness effect after SNV

Before SNV normalisation, PC1 explained 82.31% of the variance and was almost perfectly correlated with mean reflectance.

After SNV, the variance is distributed across more PCA components. We now check the correlation between the new PCA components and the original mean reflectance to see whether the brightness effect has been reduced.

In [ ]:
# Check correlation between SNV-PCA components
# and the original mean reflectance

print("Correlation with original mean reflectance after SNV:")

for i in range(spectra_pca_snv.shape[1]):

    correlation = np.corrcoef(
        spectra_pca_snv[:, i],
        mean_reflectance
    )[0, 1]

    print(f"PC{i + 1}: {correlation:.3f}")

### PCA comparison

Before SNV normalisation, PC1 explained 82.31% of the variance and had a correlation of 1.000 with mean reflectance. This showed that the first PCA representation was strongly dominated by overall spectral brightness.

After SNV normalisation, five principal components retain 96.34% of the total variance. The correlations between the PCA components and the original mean reflectance are substantially lower.

For the initial clustering, the SNV-normalised PCA representation is therefore used so that clustering is influenced more by spectral shape and less by overall brightness.

## Choose the number of clusters

K-Means requires the number of clusters (`K`) to be specified in advance.

To help choose a suitable value, K-Means is tested for several values of `K`. For each value, the within-cluster sum of squared distances (inertia) is calculated.

As `K` increases, inertia will always decrease. We look for an **elbow point**, where adding more clusters gives only a relatively small improvement.

The clustering is performed on the five PCA components obtained from the SNV-normalised spectra.

In [ ]:
# Test different numbers of clusters using the elbow method

from sklearn.cluster import KMeans

k_values = range(2, 11)
inertias = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    kmeans.fit(spectra_pca_snv)

    inertias.append(kmeans.inertia_)

    print(f"K = {k}: inertia = {kmeans.inertia_:,.2f}")


# Plot the elbow curve
plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Choosing K")

plt.xticks(list(k_values))
plt.grid(alpha=0.2)
plt.tight_layout()

plt.show()

## Check cluster separation with silhouette score

The elbow method suggests that around 4–5 clusters may provide a reasonable balance between simplicity and cluster fit.

To check this further, the silhouette score is calculated for different values of `K`. The silhouette score measures how similar each spectrum is to its own cluster compared with the other clusters.

A higher silhouette score indicates better-separated and more compact clusters.

Because the dataset contains 18,508 spectra, a reproducible sample of 5,000 spectra is used for this diagnostic to keep the calculation manageable.

In [ ]:
# Compare different K values using silhouette score

from sklearn.metrics import silhouette_score

k_values = range(2, 11)
silhouette_scores = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    cluster_labels = kmeans.fit_predict(spectra_pca_snv)

    score = silhouette_score(
        spectra_pca_snv,
        cluster_labels,
        sample_size=5000,
        random_state=42
    )

    silhouette_scores.append(score)

    print(f"K = {k}: silhouette score = {score:.4f}")


# Plot silhouette scores
plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score for Choosing K")

plt.xticks(list(k_values))
plt.grid(alpha=0.2)
plt.tight_layout()

plt.show()

### Initial choice of K

The silhouette score is highest for `K = 2`, indicating that two broad groups are the most clearly separated.

However, the elbow method suggests that approximately 4–5 clusters may capture additional structure in the spectral data. Among these values, `K = 4` has a higher silhouette score than `K = 5`.

Therefore, `K = 4` is selected for the initial clustering. This provides more spectral detail than two broad clusters while maintaining reasonable cluster separation.

In [ ]:
# Run the initial K-Means clustering with K = 4

final_k = 4

kmeans_final = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans_final.fit_predict(spectra_pca_snv)

print("Number of spectra clustered:", f"{len(cluster_labels):,}")
print("\nCluster sizes:")

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()

for cluster, count in cluster_counts.items():
    percentage = count / len(cluster_labels) * 100

    print(
        f"Cluster {cluster}: "
        f"{count:,} spectra "
        f"({percentage:.1f}%)"
    )

## Inspect the spectral shape of each cluster

K-Means assigned all 18,508 retained spectra to four clusters.

The cluster numbers are arbitrary and do not yet represent mineral types. To understand what separates the clusters, the average SNV-normalised spectrum is calculated for each cluster and plotted across wavelength.

Because clustering was performed using the SNV-normalised spectra, this plot shows the spectral-shape differences that influenced the clustering.

In [ ]:
# Plot the average SNV spectrum for each cluster

wavelengths = np.asarray(nir.wavelength)

plt.figure(figsize=(12, 6))

for cluster in range(final_k):

    cluster_mask = cluster_labels == cluster

    mean_cluster_spectrum = spectra_snv[
        cluster_mask
    ].mean(axis=0)

    plt.plot(
        wavelengths,
        mean_cluster_spectrum,
        label=f"Cluster {cluster}"
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("Mean SNV-normalised reflectance")
plt.title(f"Average Spectral Shape by Cluster — {hole_id}")

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.show()

## Compare cluster shapes in the SWIR region

The full spectral plot shows that the four clusters have a similar overall shape, but there are differences in several wavelength regions.

To examine these differences more clearly, the average cluster spectra are plotted only between 1300 nm and 2500 nm.

This region contains several important absorption features and makes it easier to see how the clusters differ in spectral shape.

In [ ]:
# Zoom into the SWIR region: 1300–2500 nm

wavelengths = np.asarray(nir.wavelength)

swir_mask = (
    (wavelengths >= 1300) &
    (wavelengths <= 2500)
)

plt.figure(figsize=(12, 6))

for cluster in range(final_k):

    cluster_mask = cluster_labels == cluster

    mean_cluster_spectrum = spectra_snv[
        cluster_mask
    ].mean(axis=0)

    plt.plot(
        wavelengths[swir_mask],
        mean_cluster_spectrum[swir_mask],
        label=f"Cluster {cluster}"
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("Mean SNV-normalised reflectance")
plt.title(f"Average Cluster Spectra — SWIR Region — {hole_id}")

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.show()

## Cluster distribution with depth

The average SWIR spectra show clear differences between the four clusters, particularly around several absorption regions.

The next step is to attach the cluster labels back to their original drill-hole depths. This allows us to see whether the spectral clusters occur in continuous depth intervals or are scattered randomly throughout the hole.

The cluster numbers are mathematical labels only and do not yet represent specific minerals.

In [ ]:
# Attach cluster labels back to the original depth information

cluster_depth = metadata_kept.copy()

cluster_depth["cluster"] = cluster_labels

cluster_depth["depth_m"] = pd.to_numeric(
    cluster_depth["depth_m"],
    errors="coerce"
)

cluster_depth = cluster_depth.sort_values("depth_m")

print("Number of clustered readings:", len(cluster_depth))

display(cluster_depth.head(20))

## Visualise clusters along the drill hole

Each retained spectrum now has a cluster label and its original drill-hole depth.

The cluster assignments are plotted against depth to examine whether particular spectral patterns form continuous intervals or occur throughout the hole.

Continuous or repeated depth intervals may indicate meaningful changes in spectral behaviour along the drill core.

In [ ]:
# Plot cluster assignments against depth

plt.figure(figsize=(14, 5))

plt.scatter(
    cluster_depth["depth_m"],
    cluster_depth["cluster"],
    c=cluster_depth["cluster"],
    cmap="tab10",
    s=5,
    alpha=0.7
)

plt.xlabel("Depth (m)")
plt.ylabel("Cluster")
plt.title(f"Spectral Cluster Distribution by Depth — {hole_id}")

plt.yticks(range(final_k))
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Cluster proportions by depth interval

The individual cluster assignments are highly detailed because each HyLogger reading represents a very small depth interval.

To reveal broader patterns along the drill hole, the readings are grouped into 1 m depth bins. For each depth interval, the proportion of readings belonging to each cluster is calculated.

This allows us to identify intervals where particular spectral clusters become more common or dominant.

In [ ]:
# Calculate cluster proportions within 1 m depth intervals

cluster_depth["depth_bin"] = np.floor(
    cluster_depth["depth_m"]
).astype(int)

cluster_proportions = pd.crosstab(
    cluster_depth["depth_bin"],
    cluster_depth["cluster"],
    normalize="index"
)

print("Cluster proportions by 1 m depth interval:")

display(cluster_proportions.head(15))

In [ ]:
# Plot cluster proportions down the drill hole

plt.figure(figsize=(14, 6))

for cluster in range(final_k):

    plt.plot(
        cluster_proportions.index,
        cluster_proportions[cluster],
        label=f"Cluster {cluster}"
    )

plt.xlabel("Depth (m)")
plt.ylabel("Proportion of readings")
plt.title(f"Cluster Proportions by 1 m Depth Interval — {hole_id}")

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.show()

## Find the dominant cluster in each depth interval

The individual HyLogger readings are very closely spaced, so the cluster assignments are summarised using 1 m depth intervals.

For each interval, the most common cluster is identified as the dominant cluster. The proportion of readings belonging to that cluster is also calculated.

This produces a simpler depth-based summary of the spectral clustering.

In [ ]:
# Find the dominant cluster within each 1 m depth interval

# Count readings from each cluster in every 1 m bin
cluster_counts_1m = pd.crosstab(
    cluster_depth["depth_bin"],
    cluster_depth["cluster"]
)

# Dominant cluster in each interval
dominant_cluster = cluster_counts_1m.idxmax(axis=1)

# Number of readings belonging to the dominant cluster
dominant_count = cluster_counts_1m.max(axis=1)

# Total number of readings in each interval
total_count = cluster_counts_1m.sum(axis=1)

# Percentage belonging to the dominant cluster
dominant_percentage = (
    dominant_count / total_count * 100
)

# Build interval summary table
depth_summary = pd.DataFrame({
    "start_depth_m": cluster_counts_1m.index,
    "end_depth_m": cluster_counts_1m.index + 1,
    "dominant_cluster": dominant_cluster,
    "dominant_percentage": dominant_percentage.round(1),
    "total_readings": total_count
})

display(depth_summary.head(20))

## Build continuous cluster intervals

The clustering results are currently summarised in 1 m depth bins.

For the initial interpretation, a depth bin is assigned to a cluster when at least 60% of its readings belong to that cluster. Bins below this threshold are labelled as `MIXED`.

Neighbouring 1 m bins with the same label are then merged to create longer continuous depth intervals.

The 60% threshold is an initial interpretation rule and can be adjusted later.

In [ ]:
# Build continuous depth intervals from the 1 m cluster summary

confidence_threshold = 60.0

interval_bins = depth_summary.copy()

# Label confident bins with their dominant cluster;
# otherwise label them as MIXED
interval_bins["interval_label"] = np.where(
    interval_bins["dominant_percentage"] >= confidence_threshold,
    "Cluster " + interval_bins["dominant_cluster"].astype(str),
    "MIXED"
)

# Start a new segment whenever the label changes
interval_bins["segment_id"] = (
    interval_bins["interval_label"]
    != interval_bins["interval_label"].shift()
).cumsum()

# Merge consecutive bins with the same label
cluster_intervals = (
    interval_bins
    .groupby("segment_id", as_index=False)
    .agg(
        start_depth_m=("start_depth_m", "min"),
        end_depth_m=("end_depth_m", "max"),
        interval_label=("interval_label", "first"),
        mean_confidence_pct=("dominant_percentage", "mean"),
        total_readings=("total_readings", "sum")
    )
)

cluster_intervals["length_m"] = (
    cluster_intervals["end_depth_m"]
    - cluster_intervals["start_depth_m"]
)

cluster_intervals["mean_confidence_pct"] = (
    cluster_intervals["mean_confidence_pct"].round(1)
)

display(cluster_intervals)

## Check the length of the cluster intervals

The 1 m cluster summaries have been merged into 67 continuous intervals.

Before interpreting the clusters using the TSG mineral classifications, the interval lengths are checked to see whether the result contains many very short segments.

A large number of 1 m intervals may indicate that the clustering is still quite locally variable, while longer continuous intervals provide stronger evidence of broader spectral zones.

In [ ]:
# Summarise the lengths of the continuous intervals

print("Total continuous intervals:", len(cluster_intervals))

print("\nInterval length summary:")
print(cluster_intervals["length_m"].describe())

print("\nNumber of intervals by length:")
print(
    cluster_intervals["length_m"]
    .value_counts()
    .sort_index()
)

print("\nNumber of intervals by label:")
print(
    cluster_intervals["interval_label"]
    .value_counts()
)

## Test 2 m depth intervals

The initial 1 m interval summary produced 67 continuous segments, and 35 of these segments were only 1 m long.

This indicates that the 1 m summary may be too sensitive to local changes in cluster composition.

To examine broader spectral zones, the same cluster assignments are now summarised using 2 m depth intervals. The clustering itself is unchanged; only the depth-based summary is made less granular.

In [ ]:
# Summarise the existing cluster assignments using 2 m depth bins

cluster_depth["depth_bin_2m"] = (
    np.floor(cluster_depth["depth_m"] / 2) * 2
)

# Count clusters inside each 2 m interval
cluster_counts_2m = pd.crosstab(
    cluster_depth["depth_bin_2m"],
    cluster_depth["cluster"]
)

# Find dominant cluster
dominant_cluster_2m = cluster_counts_2m.idxmax(axis=1)

# Number of readings in dominant cluster
dominant_count_2m = cluster_counts_2m.max(axis=1)

# Total readings
total_count_2m = cluster_counts_2m.sum(axis=1)

# Dominant percentage
dominant_percentage_2m = (
    dominant_count_2m / total_count_2m * 100
)

# Build summary
depth_summary_2m = pd.DataFrame({
    "start_depth_m": cluster_counts_2m.index,
    "end_depth_m": cluster_counts_2m.index + 2,
    "dominant_cluster": dominant_cluster_2m,
    "dominant_percentage": dominant_percentage_2m.round(1),
    "total_readings": total_count_2m
})

display(depth_summary_2m)

## Evaluate the 2 m interval summary

The 2 m summary contains 83 depth bins across the approximately 166 m scanned section.

Some intervals have a clearly dominant cluster, while others remain highly mixed. To distinguish these cases, the same 60% confidence threshold used for the 1 m analysis is applied.

Neighbouring 2 m bins with the same confident cluster label are then merged. This allows the 1 m and 2 m summaries to be compared before selecting an interval size.

In [ ]:
# Apply the 60% confidence rule to the 2 m bins

confidence_threshold = 60.0

interval_bins_2m = depth_summary_2m.copy()

interval_bins_2m["interval_label"] = np.where(
    interval_bins_2m["dominant_percentage"] >= confidence_threshold,
    "Cluster " + interval_bins_2m["dominant_cluster"].astype(str),
    "MIXED"
)

# Start a new segment whenever the label changes
interval_bins_2m["segment_id"] = (
    interval_bins_2m["interval_label"]
    != interval_bins_2m["interval_label"].shift()
).cumsum()

# Merge neighbouring bins with the same label
cluster_intervals_2m = (
    interval_bins_2m
    .groupby("segment_id", as_index=False)
    .agg(
        start_depth_m=("start_depth_m", "min"),
        end_depth_m=("end_depth_m", "max"),
        interval_label=("interval_label", "first"),
        mean_confidence_pct=("dominant_percentage", "mean"),
        total_readings=("total_readings", "sum")
    )
)

cluster_intervals_2m["length_m"] = (
    cluster_intervals_2m["end_depth_m"]
    - cluster_intervals_2m["start_depth_m"]
)

cluster_intervals_2m["mean_confidence_pct"] = (
    cluster_intervals_2m["mean_confidence_pct"].round(1)
)

print("Continuous intervals using 2 m bins:",
      len(cluster_intervals_2m))

display(cluster_intervals_2m)

## Compare spectral clusters with TSG mineral classifications

The 1 m and 2 m depth summaries show that some parts of the hole are dominated by a single spectral cluster, while other sections contain a mixture of clusters.

Before selecting a final depth-summary interval, the clusters are compared with the mineral classifications produced by TSG.

This does not use the mineral labels to create the clusters. The mineral information is added only after clustering to help interpret what spectral patterns are common within each cluster.

In [ ]:
# Attach TSG mineral classifications to the clustered spectra

# Keep scalar rows corresponding to Final Mask = 1
scalars_kept = scalars.loc[keep_mask].reset_index(drop=True)

cluster_minerals = pd.DataFrame({
    "depth_m": metadata_kept["depth_m"].values,
    "cluster": cluster_labels,

    "Min1_sTSAS": scalars_kept["Min1 sTSAS"].values,
    "Grp1_sTSAS": scalars_kept["Grp1 sTSAS"].values,

    "Min1_uTSAS": scalars_kept["Min1 uTSAS"].values,
    "Grp1_uTSAS": scalars_kept["Grp1 uTSAS"].values
})

print("Rows:", len(cluster_minerals))

display(cluster_minerals.head(20))

## Compare neighbouring spectra with the same mineral label

Some neighbouring readings have the same primary TSG mineral classification but were assigned to different spectral clusters.

Because K-Means uses the full spectral shape rather than the mineral label, the corresponding spectra are compared directly to determine how their shapes differ.

In [ ]:
# Compare nearby spectra directly

rows_to_compare = [0, 1, 2, 5]

plt.figure(figsize=(12, 6))

for row in rows_to_compare:

    depth = float(metadata_kept.loc[row, "depth_m"])
    cluster = cluster_labels[row]
    mineral = cluster_minerals.loc[row, "Min1_sTSAS"]

    # Show missing mineral classification clearly
    if pd.isna(mineral):
        mineral = "UNCLASSIFIED"

    plt.plot(
        nir.wavelength,
        spectra_snv[row],
        label=f"{depth:.3f} m | Cluster {cluster} | {mineral}"
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("SNV-normalised reflectance")
plt.title("Comparison of Nearby Spectra")

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.show()

## Mineral groups within each cluster

The clusters were created only from the spectral measurements and did not use the TSG mineral classifications.

To help interpret the clusters, the `Grp1 sTSAS` mineral group is now compared with each cluster.

Missing mineral classifications are labelled as `UNCLASSIFIED` so that they are not silently excluded.

In [ ]:
# Compare clusters with sTSAS mineral groups

cluster_group_check = cluster_minerals.copy()

# Use the same label for missing / empty values
cluster_group_check["Grp1_sTSAS"] = (
    cluster_group_check["Grp1_sTSAS"]
    .astype("string")
    .str.strip()
)

cluster_group_check["Grp1_sTSAS"] = (
    cluster_group_check["Grp1_sTSAS"]
    .mask(
        cluster_group_check["Grp1_sTSAS"].isna()
        | (cluster_group_check["Grp1_sTSAS"] == ""),
        "BLANK"
    )
)

# Percentage of each mineral group inside each cluster
group_percentages = pd.crosstab(
    cluster_group_check["cluster"],
    cluster_group_check["Grp1_sTSAS"],
    normalize="index"
) * 100

# Print the 5 most common groups in each cluster
for cluster in range(final_k):

    print(f"\nCluster {cluster}")

    top_groups = (
        group_percentages.loc[cluster]
        .sort_values(ascending=False)
        .head(5)
    )

    for group, percentage in top_groups.items():
        print(f"  {group}: {percentage:.1f}%")

## Check HyLogger diagnostic quality flags

The initial clustering removed readings where `Final Mask = 0`.

The `HyLogDiag` scalar column contains an additional diagnostic quality flag.  
This is checked to determine whether spectra retained by the Final Mask include readings with diagnostic conditions other than `Good`.

This step is only a quality check. The diagnostic flag is not used as an input to K-Means clustering.

In [ ]:
# Check HyLogDiag for the spectra used in clustering

hylogdiag_kept = scalars.loc[keep_mask, "HyLogDiag"].reset_index(drop=True)

diag_counts = hylogdiag_kept.value_counts(dropna=False)
diag_percentages = hylogdiag_kept.value_counts(
    normalize=True,
    dropna=False
) * 100

diag_summary = pd.DataFrame({
    "count": diag_counts,
    "percentage": diag_percentages.round(2)
})

display(diag_summary)

## Compare clusters with uTSAS mineral groups

The clusters were created using the SNV-normalised spectra.

The same clusters are now compared with the `Grp1 uTSAS` mineral classifications.

The vocabulary file indicates that `uTSAS` represents TSG results based on user/geologist-refined mineral selection, while `sTSAS` represents the automatic system interpretation.

The mineral classifications are used only to interpret the clusters and were not used to create them.

In [ ]:
# Compare clusters with uTSAS mineral groups

cluster_utsas_check = cluster_minerals.copy()

# Use the same label for missing / empty values
cluster_utsas_check["Grp1_uTSAS"] = (
    cluster_utsas_check["Grp1_uTSAS"]
    .astype("string")
    .str.strip()
)

cluster_utsas_check["Grp1_uTSAS"] = (
    cluster_utsas_check["Grp1_uTSAS"]
    .mask(
        cluster_utsas_check["Grp1_uTSAS"].isna()
        | (cluster_utsas_check["Grp1_uTSAS"] == ""),
        "BLANK"
    )
)

# Percentage of each uTSAS mineral group inside each cluster
utsas_percentages = pd.crosstab(
    cluster_utsas_check["cluster"],
    cluster_utsas_check["Grp1_uTSAS"],
    normalize="index"
) * 100

# Show the five most common groups in each cluster
for cluster in range(final_k):

    print(f"\nCluster {cluster}")

    top_groups = (
        utsas_percentages.loc[cluster]
        .sort_values(ascending=False)
        .head(5)
    )

    for group, percentage in top_groups.items():
        print(f"  {group}: {percentage:.1f}%")

## Compare sTSAS and uTSAS mineral groups

The sTSAS and uTSAS mineral-group distributions are now compared directly for each spectral cluster.

Both classifications use the same labels, including `BLANK` and `INVALID`.

This comparison helps identify where the automatic system interpretation (`sTSAS`) and the user/geologist-refined interpretation (`uTSAS`) agree or differ.

The mineral classifications are used only for interpreting the clusters and were not used to create the K-Means clusters.

In [ ]:
# Compare sTSAS and uTSAS side by side

# Get every mineral-group label appearing in either result
all_groups = sorted(
    set(group_percentages.columns)
    | set(utsas_percentages.columns)
)

comparison_rows = []

for cluster in range(final_k):

    for group in all_groups:

        stsas_value = (
            group_percentages.loc[cluster, group]
            if group in group_percentages.columns
            else 0
        )

        utsas_value = (
            utsas_percentages.loc[cluster, group]
            if group in utsas_percentages.columns
            else 0
        )

        comparison_rows.append({
            "cluster": cluster,
            "mineral_group": group,
            "sTSAS_%": round(stsas_value, 1),
            "uTSAS_%": round(utsas_value, 1),
            "difference_%": round(abs(stsas_value - utsas_value), 1)
        })

stsas_utsas_comparison = pd.DataFrame(comparison_rows)

display(stsas_utsas_comparison)